In [1]:
import torch
import json

jsonl_file = "test_IG_output.jsonl"
batch_records = []
with open(jsonl_file, 'r') as f:
    for line in f:
        data = json.loads(line)
        if data.get("record_type") == "batch":
            batch_records.append(data)
            

In [78]:
import numpy as np


'''
{'record_type': 'batch',
 'batch_index': 0,
 'num_kept_samples': 32,
 'num_true_positive_samples': 0,
 'num_true_negative_samples': 32,
 'true_positive_total_attr_embed_file': None,
 'true_positive_total_attr_embed_shape': None,
 'true_positive_embedding_sample_numbers': [],
 'true_negative_total_attr_embed_file': 'tests/test_IG_embeddings/batch_000_true_negative_total_attr_embed.pt',
 'true_negative_total_attr_embed_shape': [32, 1662, 512],
 'true_negative_embedding_sample_numbers': [0,
 
 Assumption is that the order of the samples in inside the batch jsonl of the test_IG_output file will match exactly, the batch indexing
 
 1. True north is batch sample_numbers["true_positive_embedding_sample_numbers"] AND
    batch["true_negative_embedding_sample_numbers"]
    
 2. process each tp and tn individually, get batch indices
 
 3. index inputs in batch pt corresponding to tn and tp
 
 4. divide out corresponding dx to get unit signal embeddings.  Sum across H, then add unit signal to respective sample id in JSON.  We want to store the unit signal embedding for later species virulence kmeans?  yea that could be cool.
 
'''
batch_num = 24

page_offset = batch_num * 32

# sample_idxs
kept_samples_pos = torch.tensor(batch_records[batch_num]['true_positive_embedding_sample_numbers'], dtype=torch.int)
kept_samples_neg = torch.tensor(batch_records[batch_num]['true_negative_embedding_sample_numbers'], dtype=torch.int)

# kept_sample_numbers, _ = torch.sort(torch.concat([kept_samples_pos, kept_samples_neg], axis=0), dim=0)
# print('kept_sample_nums', kept_sample_numbers)
# print(len(kept_sample_numbers))

# process tp for batch
tp_batch_indices = kept_samples_pos - page_offset

# proccess tn for batch
tn_batch_indices = kept_samples_neg - page_offset
print('kept_samples_pos', kept_samples_pos)
print('kept_samples_neg', kept_samples_neg)
print('tp_batch_indices', tp_batch_indices)
print('tn_batch_indices', tn_batch_indices)
print('batch num tp samples', batch_records[batch_num]['num_true_positive_samples'])
print('batch num tn samples', batch_records[batch_num]['num_true_negative_samples'])

assert len(tp_batch_indices) == batch_records[batch_num]['num_true_positive_samples']

assert len(tn_batch_indices) == batch_records[batch_num]['num_true_negative_samples']

# total_batch_indices = batch_records[batch_num]['num_true_positive_samples'] + batch_records[batch_num]['num_true_negative_samples']

# print('total batch indices', total_batch_indices)
# print('batch num_kept_samples', batch_records[batch_num]['num_kept_samples'])
# print('len batch indices calculated', len(batch_indices))
# # assert len is same 
# assert total_batch_indices == batch_records[batch_num]['num_kept_samples'] == len(batch_indices)

# print('batch indices ', batch_indices)

# load


kept_samples_pos tensor([771, 772, 773, 774, 775, 776, 777, 781, 782, 783, 784, 785, 786, 787,
        788, 789, 790, 791, 792, 793, 794, 795, 796, 797, 798, 799],
       dtype=torch.int32)
kept_samples_neg tensor([768, 769, 770, 779, 780], dtype=torch.int32)
tp_batch_indices tensor([ 3,  4,  5,  6,  7,  8,  9, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23,
        24, 25, 26, 27, 28, 29, 30, 31], dtype=torch.int32)
tn_batch_indices tensor([ 0,  1,  2, 11, 12], dtype=torch.int32)
batch num tp samples 26
batch num tn samples 5


In [ ]:
for batch in batch_records:
    

In [61]:

# load the pt files 
file_path = f"test_IG_embeddings/batch_{batch_num:03d}_true_positive_total_attr_embed.pt"
tp_tensor = torch.load(file_path, map_location="cpu")
print(tp_tensor.shape)

file_path = f"test_IG_embeddings/batch_{batch_num:03d}_true_negative_total_attr_embed.pt"
tp_tensor = torch.load(file_path, map_location="cpu")
print(tp_tensor.shape)


torch.Size([26, 1662, 512])
torch.Size([5, 1662, 512])


In [ ]:
for batch_idx in range(234):
    file_path = f"test_species_embeddings/batch_{batch_idx:03d}_species_embeddings.pt"
    species_tensor = torch.load(file_path, map_location="cpu")
    print(species_tensor.shape)
  
    for i in range(32):
        for j in range(32):
            # asserting that every special sample embedding is the same
            # comparing matrices [0,1,...,1662] == [0,1,...,1662] where each element is actually an embedding R^512 for token 0,1,...,1662
            assert torch.equal(species_tensor[i,:,:], species_tensor[j,:,:])
   

torch.Size([32, 1662, 512])
tensor([-1.2774e-02,  5.1309e-04, -3.2695e-02, -6.5155e-03, -1.3006e-02,
         2.9354e-02,  2.3813e-02, -3.5865e-02, -5.6028e-02,  5.0902e-02,
        -4.2037e-02,  2.9468e-02,  2.0516e-02, -2.9723e-04,  5.9873e-02,
         1.5658e-02,  4.8873e-02,  3.7854e-03, -1.2064e-03, -2.8288e-02,
        -1.4271e-02,  2.0888e-02, -1.3105e-02,  1.1954e-02, -2.5131e-02,
        -6.7698e-03,  1.7232e-02,  1.2355e-02, -1.1674e-02,  3.5853e-02,
         2.1301e-02, -3.3865e-02, -2.5054e-02, -1.9135e-02,  3.6437e-02,
         8.4903e-03, -2.9463e-02,  2.1185e-03,  2.8105e-02, -1.0706e-02,
        -5.8480e-03, -4.6738e-02,  1.0213e-02,  2.3489e-04, -5.0928e-03,
         3.3880e-02, -2.3407e-02, -1.1364e-02,  5.2851e-02,  5.4038e-02,
         4.0751e-02,  3.1926e-02,  1.1409e-02, -1.1873e-02,  7.3778e-02,
        -1.6578e-02,  7.5271e-03,  1.6268e-02, -2.8323e-02,  3.4346e-02,
        -3.8881e-03,  5.0207e-02,  2.3220e-02, -7.1143e-02, -2.2511e-02,
         7.8331e-03,  8

In [43]:
for batch in batch_records:
    sample_num = batch['samples'][0]['sample_number'] - 1
    for sample in batch['samples']:
        if sample_num != sample['sample_number'] - 1:
            print('skipped sample')
            print('batch index', batch['batch_index'])
            print('between sample', sample_num)
            print('and sample', sample['sample_number'])
            break
        else:
            sample_num = sample['sample_number']

skipped sample
batch index 19
between sample 620
and sample 622
skipped sample
batch index 24
between sample 777
and sample 779
skipped sample
batch index 26
between sample 856
and sample 858
skipped sample
batch index 30
between sample 975
and sample 977
skipped sample
batch index 41
between sample 1336
and sample 1338
skipped sample
batch index 43
between sample 1399
and sample 1401
skipped sample
batch index 47
between sample 1524
and sample 1526
skipped sample
batch index 63
between sample 2042
and sample 2044
skipped sample
batch index 72
between sample 2319
and sample 2321
skipped sample
batch index 73
between sample 2362
and sample 2364
skipped sample
batch index 74
between sample 2383
and sample 2385
skipped sample
batch index 77
between sample 2493
and sample 2495
skipped sample
batch index 78
between sample 2524
and sample 2526
skipped sample
batch index 79
between sample 2542
and sample 2544
skipped sample
batch index 84
between sample 2690
and sample 2692
skipped sample
bat

In [53]:
for batch in batch_records:
    if batch['num_true_positive_samples'] > 0 and batch['num_true_negative_samples'] > 0:
        print('batch index',batch['batch_index'])
    # sample_num = batch['samples'][0]['sample_number'] - 1
    # for sample in batch['samples']:
    #     if sample_num != sample['sample_number'] - 1:
    #         print('skipped sample')
    #         print('batch index', batch['batch_index'])
    #         print('between sample', sample_num)
    #         print('and sample', sample['sample_number'])
    #         break
    #     else:
    #         sample_num = sample['sample_number']

batch index 18
batch index 23
batch index 24
batch index 25
batch index 28
batch index 32
batch index 33
batch index 35
batch index 37
batch index 38
batch index 40
batch index 43
batch index 45
batch index 46
batch index 47
batch index 48
batch index 49
batch index 50
batch index 51
batch index 53
batch index 56
batch index 57
batch index 58
batch index 59
batch index 60
batch index 61
batch index 62
batch index 63
batch index 64
batch index 65
batch index 66
batch index 68
batch index 69
batch index 70
batch index 71
batch index 85
batch index 88
batch index 98
batch index 100
batch index 101
batch index 102
batch index 103
batch index 131
batch index 142
batch index 165
batch index 170
batch index 171
batch index 173
batch index 177
batch index 180
batch index 182
batch index 185
batch index 188
batch index 189
batch index 190
batch index 191
batch index 192
batch index 193
batch index 194
batch index 195
batch index 196
batch index 197
batch index 198
batch index 200
batch index 20